In [2]:
import os
import json
import csv
from pathlib import Path
from collections import defaultdict
from semanticher.data import load_label_cnp

In [3]:
# =========================
# Base folders
# =========================
REPO_ROOT       = Path().resolve().parents[1]
BASE_RESULT_DIR = REPO_ROOT / "results" / "main_msv_cnp"
CANDIDATE_DIR   = BASE_RESULT_DIR / "Result_msv_cnp" / "Candidate_msv_cnp"
RESULT_DIR      = BASE_RESULT_DIR / "Result_msv_cnp" / "msv_cnp_threshold"
EVAL_DIR        = BASE_RESULT_DIR / "Eval_msv_cnp"   / "msv_cnp_threshold"

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,   exist_ok=True)

df_labels = load_label_cnp()


# =========================
# Model aliases / experiments
# =========================
MODEL_ALIAS = {
    "deepseek-chat":    "dsk",
    "gemini-2.0-flash": "gem",
    "gpt-4.1-mini":     "gpt",
}

SINGLE_EXPERIMENTS = [
    ["deepseek-chat"],
    ["gemini-2.0-flash"],
    ["gpt-4.1-mini"],
]

MULTI_EXPERIMENTS = [
    ["deepseek-chat", "gemini-2.0-flash"],
    ["deepseek-chat", "gpt-4.1-mini"],
    ["gemini-2.0-flash", "gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash", "gpt-4.1-mini"],
]

EVAL_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


def build_experiment_tag(model_names):
    aliases = [MODEL_ALIAS[m] for m in model_names]
    prefix  = {1: "single", 2: "two", 3: "three"}[len(model_names)]
    return f"{prefix}_{'_'.join(aliases)}"


def build_candidate_path(model_names):
    tag = build_experiment_tag(model_names)
    return CANDIDATE_DIR / f"candidate_msv_cnp_{tag}.json"


def build_result_dir(model_names):
    tag     = build_experiment_tag(model_names)
    out_dir = RESULT_DIR / f"msv_cnp_threshold_{tag}"
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def build_eval_dir(model_names):
    tag     = build_experiment_tag(model_names)
    out_dir = EVAL_DIR / f"msv_cnp_threshold_{tag}"
    os.makedirs(out_dir, exist_ok=True)
    return out_dir

In [4]:
# =========================
# Helper functions
# =========================
def clean_label(x):
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s in ("", "-", "nan", "NaN") else s


def pred_name(item):
    if item is None:
        return ""
    if isinstance(item, (list, tuple)):
        item = item[0] if len(item) else ""
    return clean_label(item)


def to_node(x):
    if not x:
        return ""
    return str(x).split("/")[-1].strip()


def prf_only(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall    = tp / (tp + fn) if tp + fn else 0.0
    f1        = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

In [5]:
# =========================
# Ground truth & evaluation
# =========================
def build_gt(label_row):
    class_path, class_node = [], []
    for cnp in ["Class_1", "Class_2"]:
        v = clean_label(label_row.get(cnp))
        if v:
            class_path.append(v)
            class_node.append(to_node(v))

    prop_node = []
    for cnp in ["Property_1", "Property_2", "Property_3"]:
        v = clean_label(label_row.get(cnp))
        if v:
            prop_node.append(v)

    return {
        "class_path": class_path,
        "class_node": class_node,
        "prop_node":  prop_node,
    }


def prepare_eval_data(result_pred, df_labels, threshold):
    eval_json = []

    for r in result_pred:
        table_id  = r["table_id"]
        column_id = r["column_id"]

        label_row = df_labels[
            (df_labels["table_id"].astype(str)  == str(table_id)) &
            (df_labels["column_id"].astype(int) == int(column_id))
        ].iloc[0]

        pred_class = []
        for x in (r.get("class", []) or []):
            if isinstance(x, (list, tuple)) and len(x) >= 2:
                name, conf = x[0], x[1]
                try:
                    conf = float(conf)
                except Exception:
                    continue
                if conf >= threshold:
                    name = pred_name(name)
                    if name:
                        pred_class.append(name)

        pred_prop = []
        for x in (r.get("property", []) or []):
            if isinstance(x, (list, tuple)) and len(x) >= 2:
                name, conf = x[0], x[1]
                try:
                    conf = float(conf)
                except Exception:
                    continue
                if conf >= threshold:
                    name = pred_name(name)
                    if name:
                        pred_prop.append(name)

        eval_json.append({
            "table_id":    table_id,
            "table_name":  r["table_name"],
            "column_id":   column_id,
            "column_name": r["column_name"],
            "pred": {
                "class": pred_class,
                "prop":  pred_prop,
            },
            "gt": build_gt(label_row),
        })

    return eval_json


def compute_metrics_for_key(row, key, *, mode="path"):
    pred = [str(x).strip() for x in row["pred"].get(key, []) if str(x).strip()]

    if key == "class":
        gt_raw = row["gt"].get("class_path" if mode == "path" else "class_node", [])
    elif key == "prop":
        gt_raw = row["gt"].get("prop_node", [])
    else:
        raise ValueError(f"Unknown key: {key}")

    gt = [str(x).strip() for x in gt_raw if str(x).strip()]

    gt_seg_sets = []
    if key == "class" and mode == "path":
        for ans in gt:
            segs = [s.strip() for s in ans.split("/") if s.strip()]
            gt_seg_sets.append(set(segs))
    elif key == "prop":
        for ans in gt:
            segs = [s.strip() for s in ans.split("/") if s.strip()]
            gt_seg_sets.append(set(segs))
    else:
        for ans in gt:
            gt_seg_sets.append({ans})

    matched_gt_idx   = set()
    matched_pred_idx = set()

    for pi, pid in enumerate(pred):
        for gi, segs in enumerate(gt_seg_sets):
            if pid in segs:
                matched_gt_idx.add(gi)
                matched_pred_idx.add(pi)

    tp = len(matched_gt_idx)
    fp = len(pred) - len(matched_pred_idx)
    fn = len(gt_seg_sets) - len(matched_gt_idx)

    precision, recall, f1 = prf_only(tp, fp, fn)
    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}


def evaluate_c_prop_all(eval_json, mode="node"):
    row_scores_class = []
    row_scores_prop  = []
    row_scores_all   = []

    totals = {
        "class": {"tp": 0, "fp": 0, "fn": 0},
        "prop":  {"tp": 0, "fp": 0, "fn": 0},
        "all":   {"tp": 0, "fp": 0, "fn": 0},
    }

    for row in eval_json:
        res_c    = compute_metrics_for_key(row, "class", mode=mode)
        res_prop = compute_metrics_for_key(row, "prop",  mode=mode)

        tp_all = res_c["tp"] + res_prop["tp"]
        fp_all = res_c["fp"] + res_prop["fp"]
        fn_all = res_c["fn"] + res_prop["fn"]
        p_all, r_all, f1_all = prf_only(tp_all, fp_all, fn_all)

        row_scores_class.append(res_c)
        row_scores_prop.append(res_prop)
        row_scores_all.append({"tp": tp_all, "fp": fp_all, "fn": fn_all,
                               "precision": p_all, "recall": r_all, "f1": f1_all})

        for k, res in [("class", res_c), ("prop", res_prop)]:
            totals[k]["tp"] += res["tp"]
            totals[k]["fp"] += res["fp"]
            totals[k]["fn"] += res["fn"]
        totals["all"]["tp"] += tp_all
        totals["all"]["fp"] += fp_all
        totals["all"]["fn"] += fn_all

    def _macro(scores):
        n = len(scores)
        if not n:
            return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
        return {
            "precision": sum(s["precision"] for s in scores) / n,
            "recall":    sum(s["recall"]    for s in scores) / n,
            "f1":        sum(s["f1"]        for s in scores) / n,
        }

    def _micro(counts):
        p, r, f1 = prf_only(counts["tp"], counts["fp"], counts["fn"])
        return {"precision": p, "recall": r, "f1": f1}

    return {
        "macro": {
            "class": _macro(row_scores_class),
            "prop":  _macro(row_scores_prop),
            "all":   _macro(row_scores_all),
        },
        "micro": {
            "class": _micro(totals["class"]),
            "prop":  _micro(totals["prop"]),
            "all":   _micro(totals["all"]),
        },
    }


def evaluate_all_modes(eval_json):
    return {
        "path": evaluate_c_prop_all(eval_json, mode="path"),
        "node": evaluate_c_prop_all(eval_json, mode="node"),
    }


def save_eval_txt(eval_result, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("=== MODE: PATH ===\n")
        f.write("=== MACRO ===\n")
        f.write(json.dumps(eval_result["path"]["macro"], ensure_ascii=False, indent=2))
        f.write("\n")
        f.write("=== MICRO ===\n")
        f.write(json.dumps(eval_result["path"]["micro"], ensure_ascii=False, indent=2))
        f.write("\n\n")
        f.write("=== MODE: NODE ===\n")
        f.write("=== MACRO ===\n")
        f.write(json.dumps(eval_result["node"]["macro"], ensure_ascii=False, indent=2))
        f.write("\n")
        f.write("=== MICRO ===\n")
        f.write(json.dumps(eval_result["node"]["micro"], ensure_ascii=False, indent=2))
        f.write("\n")

In [6]:
# =========================
# Soft voting
# =========================
def group_candidate_rows(candidate_rows):
    grouped = defaultdict(lambda: {
        "table_id":    None,
        "table_name":  None,
        "column_name": None,
        "column_id":   None,
        "by_model":    {}
    })

    for row in candidate_rows:
        key = (row["table_id"], row["column_id"])
        grouped[key]["table_id"]    = row["table_id"]
        grouped[key]["table_name"]  = row["table_name"]
        grouped[key]["column_name"] = row["column_name"]
        grouped[key]["column_id"]   = row["column_id"]
        grouped[key]["by_model"][row["model"]] = {
            "class":    row.get("class",    []) or [],
            "property": row.get("property", []) or [],
        }

    return grouped


def soft_vote(grouped_rows, model_names):
    """
    Post mode: all candidate paths enter soft voting directly.
    class and property voted independently.
    score(label) = sum(conf_i) / N
    """
    result     = []
    num_models = len(model_names)

    for _, row in grouped_rows.items():
        class_scores = defaultdict(float)
        prop_scores  = defaultdict(float)

        for model_name in model_names:
            model_data = row["by_model"].get(model_name, {})

            for item in model_data.get("class", []):
                if not (isinstance(item, (list, tuple)) and len(item) >= 2):
                    continue
                label, conf = item[0], item[1]
                if not isinstance(label, str) or not label:
                    continue
                try:
                    conf = float(conf)
                except Exception:
                    continue
                class_scores[label] += conf / num_models

            for item in model_data.get("property", []):
                if not (isinstance(item, (list, tuple)) and len(item) >= 2):
                    continue
                label, conf = item[0], item[1]
                if not isinstance(label, str) or not label:
                    continue
                try:
                    conf = float(conf)
                except Exception:
                    continue
                prop_scores[label] += conf / num_models

        ranked_class = sorted(class_scores.items(), key=lambda x: x[1], reverse=True)
        ranked_prop  = sorted(prop_scores.items(),  key=lambda x: x[1], reverse=True)

        result.append({
            "table_id":    row["table_id"],
            "table_name":  row["table_name"],
            "column_name": row["column_name"],
            "column_id":   row["column_id"],
            "class":    [[l, round(s, 6)] for l, s in ranked_class],
            "property": [[l, round(s, 6)] for l, s in ranked_prop],
        })

    return result


def apply_threshold(voted_result, threshold):
    filtered = []

    for row in voted_result:
        filtered.append({
            "table_id":    row["table_id"],
            "table_name":  row["table_name"],
            "column_name": row["column_name"],
            "column_id":   row["column_id"],
            "class":    [[l, s] for l, s in row.get("class",    []) if s >= threshold],
            "property": [[l, s] for l, s in row.get("property", []) if s >= threshold],
        })

    return filtered


def prefilter_by_model(items, threshold):
    out = []
    for item in items:
        if not (isinstance(item, (list, tuple)) and len(item) >= 2):
            continue
        label, conf = item[0], item[1]
        if not isinstance(label, str) or not label:
            continue
        try:
            conf = float(conf)
        except Exception:
            continue
        if conf >= threshold:
            out.append((label, conf))
    return out


def soft_vote_pre(grouped_rows, model_names, best_thresholds):
    """
    Pre mode: each model's candidates filtered by its own best threshold
    before soft voting. class and property voted independently.
    score(label) = sum(conf_i) / N
    """
    result     = []
    num_models = len(model_names)

    for _, row in grouped_rows.items():
        class_scores = defaultdict(float)
        prop_scores  = defaultdict(float)

        for model_name in model_names:
            tag_single = build_experiment_tag([model_name])
            threshold  = best_thresholds[tag_single]
            model_data = row["by_model"].get(model_name, {})

            for label, conf in prefilter_by_model(model_data.get("class", []), threshold):
                class_scores[label] += conf / num_models

            for label, conf in prefilter_by_model(model_data.get("property", []), threshold):
                prop_scores[label] += conf / num_models

        ranked_class = sorted(class_scores.items(), key=lambda x: x[1], reverse=True)
        ranked_prop  = sorted(prop_scores.items(),  key=lambda x: x[1], reverse=True)

        result.append({
            "table_id":    row["table_id"],
            "table_name":  row["table_name"],
            "column_name": row["column_name"],
            "column_id":   row["column_id"],
            "class":    [[l, round(s, 6)] for l, s in ranked_class],
            "property": [[l, round(s, 6)] for l, s in ranked_prop],
        })

    return result

In [7]:
# =========================
# Step 1: single LLM threshold sweep
# =========================
def run_single_threshold_sweep():
    best_thresholds = {}

    for model_names in SINGLE_EXPERIMENTS:
        tag            = build_experiment_tag(model_names)
        candidate_file = build_candidate_path(model_names)
        result_dir     = build_result_dir(model_names)
        eval_dir       = build_eval_dir(model_names)

        print("=" * 80)
        print(f"Single sweep: {tag}")
        print("=" * 80)

        with open(candidate_file, encoding="utf-8") as f:
            candidate_rows = json.load(f)

        grouped_rows = group_candidate_rows(candidate_rows)
        voted_result = soft_vote(grouped_rows, model_names)

        scores = {}

        for threshold in EVAL_THRESHOLDS:
            result_pred = apply_threshold(voted_result, threshold)

            ttag        = str(threshold).replace(".", "")
            result_path = result_dir / f"msv_cnp_threshold_{tag}_t{ttag}.json"
            with open(result_path, "w", encoding="utf-8") as f:
                json.dump(result_pred, f, ensure_ascii=False, indent=2)

            eval_json   = prepare_eval_data(result_pred, df_labels, threshold=0)
            eval_result = evaluate_all_modes(eval_json)

            f1                = eval_result["node"]["micro"]["all"]["f1"]
            scores[threshold] = f1

            t_dir    = eval_dir / f"t{ttag}_eval"
            os.makedirs(t_dir, exist_ok=True)
            out_file = t_dir / f"msv_cnp_threshold_{tag}_t{ttag}_eval.txt"
            save_eval_txt(eval_result, out_file)

            print(f"  threshold={threshold:.1f}  node_micro_all_f1={f1:.6f}")

        best_t               = max(scores, key=scores.get)
        best_thresholds[tag] = best_t

        best_dir = eval_dir / "best_eval"
        os.makedirs(best_dir, exist_ok=True)
        best_result = apply_threshold(voted_result, best_t)
        best_eval   = evaluate_all_modes(prepare_eval_data(best_result, df_labels, threshold=0))
        best_out    = best_dir / f"msv_cnp_threshold_{tag}_best_t{str(best_t).replace('.', '')}_eval.txt"
        save_eval_txt(best_eval, best_out)

        print(f"  best threshold: {best_t}  f1={scores[best_t]:.6f}\n")

    print("BEST_THRESHOLDS:", best_thresholds)
    return best_thresholds


BEST_THRESHOLDS = run_single_threshold_sweep()

Single sweep: single_dsk
  threshold=0.3  node_micro_all_f1=0.333333
  threshold=0.4  node_micro_all_f1=0.333333
  threshold=0.5  node_micro_all_f1=0.333333
  threshold=0.6  node_micro_all_f1=0.242424
  threshold=0.7  node_micro_all_f1=0.275862
  threshold=0.8  node_micro_all_f1=0.086957
  threshold=0.9  node_micro_all_f1=0.105263
  best threshold: 0.3  f1=0.333333

Single sweep: single_gem
  threshold=0.3  node_micro_all_f1=0.380952
  threshold=0.4  node_micro_all_f1=0.380952
  threshold=0.5  node_micro_all_f1=0.421053
  threshold=0.6  node_micro_all_f1=0.470588
  threshold=0.7  node_micro_all_f1=0.615385
  threshold=0.8  node_micro_all_f1=0.631579
  threshold=0.9  node_micro_all_f1=0.266667
  best threshold: 0.8  f1=0.631579

Single sweep: single_gpt
  threshold=0.3  node_micro_all_f1=0.500000
  threshold=0.4  node_micro_all_f1=0.500000
  threshold=0.5  node_micro_all_f1=0.545455
  threshold=0.6  node_micro_all_f1=0.562500
  threshold=0.7  node_micro_all_f1=0.428571
  threshold=0.8  

In [8]:
# =========================
# Step 2: multi LLM pre-filter + soft vote
# =========================
def run_multi_pre_experiment(model_names, best_thresholds):
    tag            = build_experiment_tag(model_names)
    candidate_file = build_candidate_path(model_names)
    result_dir     = build_result_dir(model_names)
    eval_dir       = build_eval_dir(model_names)

    print("=" * 80)
    print(f"Multi pre sweep: {tag}")
    print("Models:    ", model_names)
    print("=" * 80)

    with open(candidate_file, encoding="utf-8") as f:
        candidate_rows = json.load(f)

    grouped_rows = group_candidate_rows(candidate_rows)
    voted_result = soft_vote_pre(grouped_rows, model_names, best_thresholds)

    raw_vote_path = result_dir / f"msv_cnp_threshold_{tag}_raw_vote.json"
    with open(raw_vote_path, "w", encoding="utf-8") as f:
        json.dump(voted_result, f, ensure_ascii=False, indent=2)

    scores       = {}
    full_results = {}

    print(f"\n===== threshold sweep: {tag} =====")

    for threshold in EVAL_THRESHOLDS:
        result_pred = apply_threshold(voted_result, threshold)

        ttag        = str(threshold).replace(".", "")
        result_path = result_dir / f"msv_cnp_threshold_{tag}_t{ttag}.json"
        with open(result_path, "w", encoding="utf-8") as f:
            json.dump(result_pred, f, ensure_ascii=False, indent=2)

        eval_json   = prepare_eval_data(result_pred, df_labels, threshold=0)
        eval_result = evaluate_all_modes(eval_json)

        f1                      = eval_result["node"]["micro"]["all"]["f1"]
        scores[threshold]       = f1
        full_results[threshold] = eval_result

        print(f"  threshold={threshold:.1f}  node_micro_all_f1={f1:.6f}")

    best_t = max(scores, key=scores.get)
    print(f"\n  best threshold: {best_t}  f1={scores[best_t]:.6f}")

    best_dir = eval_dir / "best_eval"
    os.makedirs(best_dir, exist_ok=True)

    for threshold in EVAL_THRESHOLDS:
        ttag     = str(threshold).replace(".", "")
        t_dir    = eval_dir / f"t{ttag}_eval"
        os.makedirs(t_dir, exist_ok=True)
        out_file = t_dir / f"msv_cnp_threshold_{tag}_t{ttag}_eval.txt"
        save_eval_txt(full_results[threshold], out_file)

    best_out = best_dir / f"msv_cnp_threshold_{tag}_best_t{str(best_t).replace('.', '')}_eval.txt"
    save_eval_txt(full_results[best_t], best_out)

    print(f"  Saved eval -> {eval_dir}")

    return scores, full_results, best_t

In [9]:
# =========================
# Run all multi + summary
# =========================
def run_all_multi_experiments(best_thresholds):
    all_scores       = {}
    all_full_results = {}
    all_best_t       = {}

    for model_names in MULTI_EXPERIMENTS:
        tag = build_experiment_tag(model_names)
        scores, full_results, best_t = run_multi_pre_experiment(model_names, best_thresholds)
        all_scores[tag]       = scores
        all_full_results[tag] = full_results
        all_best_t[tag]       = best_t

    # summary csv
    summary_csv = EVAL_DIR / "threshold_summary.csv"
    with open(summary_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["experiment"] + [f"t{str(t).replace('.','')}" for t in EVAL_THRESHOLDS] + ["best"])

        for tag in all_scores:
            row = [tag]
            for t in EVAL_THRESHOLDS:
                row.append(round(all_scores[tag][t], 6) if t in all_scores[tag] else "")
            row.append(all_best_t[tag])
            writer.writerow(row)

    # summary txt
    summary_txt = EVAL_DIR / "summary.txt"
    with open(summary_txt, "w", encoding="utf-8") as f:
        f.write("MSV CNP + Threshold sweep\n")
        f.write("Best threshold criterion: NODE / MICRO / ALL / F1\n")
        f.write(f"BEST_THRESHOLDS (single): {BEST_THRESHOLDS}\n\n")

        for tag in all_scores:
            f.write(f"{tag}\n")
            for t in EVAL_THRESHOLDS:
                f.write(f"  {t:.1f} -> {all_scores[tag][t]:.6f}\n")
            f.write(f"  best = {all_best_t[tag]}\n\n")

    print(f"Saved summary -> {summary_csv}")
    print(f"Saved summary -> {summary_txt}")


run_all_multi_experiments(BEST_THRESHOLDS)

Multi pre sweep: two_dsk_gem
Models:     ['deepseek-chat', 'gemini-2.0-flash']

===== threshold sweep: two_dsk_gem =====
  threshold=0.3  node_micro_all_f1=0.342857
  threshold=0.4  node_micro_all_f1=0.428571
  threshold=0.5  node_micro_all_f1=0.470588
  threshold=0.6  node_micro_all_f1=0.470588
  threshold=0.7  node_micro_all_f1=0.470588
  threshold=0.8  node_micro_all_f1=0.375000
  threshold=0.9  node_micro_all_f1=0.000000

  best threshold: 0.5  f1=0.470588
  Saved eval -> D:\RWTH\Github\sta-multi-llm-framework\results\main_msv_cnp\Eval_msv_cnp\msv_cnp_threshold\msv_cnp_threshold_two_dsk_gem
Multi pre sweep: two_dsk_gpt
Models:     ['deepseek-chat', 'gpt-4.1-mini']

===== threshold sweep: two_dsk_gpt =====
  threshold=0.3  node_micro_all_f1=0.400000
  threshold=0.4  node_micro_all_f1=0.437500
  threshold=0.5  node_micro_all_f1=0.333333
  threshold=0.6  node_micro_all_f1=0.333333
  threshold=0.7  node_micro_all_f1=0.363636
  threshold=0.8  node_micro_all_f1=0.000000
  threshold=0.9  